Importamos los paquetes necesarios para la ejecución del notebook

In [1]:
import time
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

Declaración y configuración de parámetros

In [2]:
FACE_MODEL = "./models/face_landmarker.task"
MASK_PATHS = ["./assets/masks/P5_Joker.png", "./assets/masks/P5_Ann.png", "./assets/masks/P5_Kasumi.png"]
MAX_FACES = 3

NOSE_DEPTH = 0.5 # cuánto sobresale la nariz respecto a los ojos (en dist. entre ojos); para estimar el giro

FEATHER = 0.8 # difuminado del borde de los sprites (sigma en px, 0 = sin)
BRIGHT_MATCH = 0.6 # ajuste de brillo del sprite al vídeo (0 = sin ajuste, 1 = completo)

# Detección y estabilidad
DETECT_EVERY = 1  # Mediapipe es rápido y en modo vídeo sigue las caras: se puede detectar en cada frame
HOLD = 3 # nº de detecciones fallidas que se mantiene el filtro antes de quitarlo
MIN_CUTOFF = 1.0 # filtro One Euro: más bajo = más suave con la cabeza quieta
BETA = 0.02 # filtro One Euro: más alto = menos retraso al moverse

Definición de funciones auxiliares de dibujado y estabilización

In [3]:
KEYS = ["left_eye", "right_eye", "nose", "mouth_left", "mouth_right"]


# Máscaras 
def load_rgba(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    assert im is not None, f"No se encuentra {path}"
    if im.shape[2] == 3:  # sin alfa: se genera quitando el blanco
        a = (~np.all(im > 240, axis=2)).astype(np.uint8) * 255
        im = np.dstack([im, a])
    return im


def prepare(im):
    """BGRA uint8 -> float32 con RGB premultiplicado por alfa (evita bordes oscuros) y borde suave."""
    f = im.astype(np.float32)
    f[:, :, :3] *= f[:, :, 3:4] / 255.0
    if FEATHER > 0:
        f = cv2.GaussianBlur(f, (0, 0), FEATHER)
    return f


def eye_holes(im):
    """Ancla y escala de una máscara: punto medio y separación de sus dos agujeros de ojos
    (las dos zonas transparentes cerradas más grandes, que no tocan el borde de la imagen)."""
    n, lab, st, cen = cv2.connectedComponentsWithStats((im[:, :, 3] < 20).astype(np.uint8))
    H, W = im.shape[:2]
    holes = [(st[i, cv2.CC_STAT_AREA], cen[i]) for i in range(1, n)
             if st[i, 0] > 0 and st[i, 1] > 0 and st[i, 0] + st[i, 2] < W and st[i, 1] + st[i, 3] < H]
    assert len(holes) >= 2, "No encuentro los dos agujeros de los ojos (¿están transparentes en el PNG?)"
    (_, c1), (_, c2) = sorted(holes, key=lambda h: -h[0])[:2]
    return (c1 + c2) / 2, float(np.linalg.norm(c1 - c2))


MASKS = []
for path in MASK_PATHS:
    _im = load_rgba(path)
    _anchor, _ref = eye_holes(_im)
    MASKS.append((prepare(_im), _anchor, _ref))


# Funciones de dibujado y geometría
def overlay(frame, sprite, src_pt, dst_pt, sx, sy, angle_deg, gain=1.0):
    """Pinta el sprite sobre frame: src_pt cae en dst_pt, con escala (sx, sy) en los ejes
    del sprite y rotación angle_deg (horaria en pantalla). Solo procesa la zona afectada."""
    a = np.radians(angle_deg)
    A = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]]) @ np.diag([sx, sy])
    t = np.asarray(dst_pt, float) - A @ np.asarray(src_pt, float)
    h, w = sprite.shape[:2]
    corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], float) @ A.T + t
    x0, y0 = np.floor(corners.min(0)).astype(int)
    x1, y1 = np.ceil(corners.max(0)).astype(int)
    H, W = frame.shape[:2]
    x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
    if x1 <= x0 or y1 <= y0:
        return
    M = np.hstack([A, (t - [x0, y0])[:, None]])
    warped = cv2.warpAffine(sprite, M, (x1 - x0, y1 - y0), flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    alpha = warped[:, :, 3:4] / 255.0
    roi = frame[y0:y1, x0:x1]
    roi[:] = np.clip(warped[:, :, :3] * gain + roi * (1 - alpha), 0, 255).astype(np.uint8)


def face_pose(kp):
    """Ojos ordenados, ángulo, punto medio, vector 'arriba', escala corregida por giro y cos(giro)."""
    le, re = kp["left_eye"], kp["right_eye"]
    if le[0] > re[0]:
        le, re = re, le
    d = re - le
    iod = np.hypot(*d)
    u = d / iod
    angle = np.degrees(np.arctan2(d[1], d[0]))
    mid = (le + re) / 2
    up = np.array([u[1], -u[0]])
    # giro de la cabeza (aprox.): desplazamiento lateral de la nariz respecto al punto medio de los ojos
    t = np.dot(kp["nose"] - mid, u) / iod
    cy = float(np.clip(1.0 / np.sqrt(1.0 + (t / NOSE_DEPTH) ** 2), 0.6, 1.0))
    return mid, up, angle, iod / cy, cy


def brightness_gain(frame, kp):
    """Brillo medio de la cara -> factor para oscurecer/aclarar los sprites."""
    mid, _, _, iod, _ = face_pose(kp)
    H, W = frame.shape[:2]
    x0, x1 = int(max(mid[0] - iod, 0)), int(min(mid[0] + iod, W))
    y0, y1 = int(max(mid[1] - 0.3 * iod, 0)), int(min(mid[1] + iod, H))
    if x1 <= x0 or y1 <= y0:
        return 1.0
    g = 1 + BRIGHT_MATCH * (frame[y0:y1, x0:x1].mean() / 140.0 - 1)
    return float(np.clip(g, 0.6, 1.0))


def apply_filter(frame, kp, mask_i, gain=1.0):
    mid, up, angle, iod, cy = face_pose(kp)
    sprite, anchor, ref = MASKS[mask_i]
    s = iod / ref                       # los agujeros de la máscara quedan sobre tus ojos
    overlay(frame, sprite, anchor, mid, s * cy, s, angle, gain)


# Estabilidad: One Euro + seguimiento
class OneEuro:
    """Suaviza sin retraso: poco cutoff con la cabeza quieta, más cutoff cuanto más rápido se mueve."""
    def __init__(self, min_cutoff=MIN_CUTOFF, beta=BETA, d_cutoff=1.0):
        self.min_cutoff, self.beta, self.d_cutoff = min_cutoff, beta, d_cutoff
        self.x = self.dx = self.t = None

    @staticmethod
    def _alpha(cutoff, dt):
        return 1.0 / (1.0 + 1.0 / (2 * np.pi * cutoff * dt))

    def __call__(self, x, t):
        if self.x is None:
            self.x, self.dx, self.t = x.copy(), np.zeros_like(x), t
            return self.x
        dt = max(t - self.t, 1e-3)
        a = self._alpha(self.d_cutoff, dt)
        self.dx = a * (x - self.x) / dt + (1 - a) * self.dx
        cutoff = self.min_cutoff + self.beta * np.linalg.norm(self.dx, axis=-1, keepdims=True)
        a = self._alpha(cutoff, dt)
        self.x = a * x + (1 - a) * self.x
        self.t = t
        return self.x


class Track:
    """Una cara seguida: landmarks suavizados, extrapolación entre detecciones y mantenimiento si falla."""
    def __init__(self):
        self.f, self.miss, self.gain, self.t = OneEuro(), 0, 1.0, 0.0
        self.mask = 0                      # índice de su máscara

    def update(self, kp, t):
        self.f(np.array([kp[k] for k in KEYS], float), t)
        self.miss, self.t = 0, t

    def kp(self, t):
        x = self.f.x
        if self.miss == 0:  # extrapolación corta (máx. 0.12 s) entre detecciones
            x = x + self.f.dx * min(max(t - self.t, 0.0), 0.12)
        return dict(zip(KEYS, x))

    def center(self):
        return (self.f.x[0] + self.f.x[1]) / 2

    def iod(self):
        return np.linalg.norm(self.f.x[0] - self.f.x[1])


def update_tracks(tracks, faces, t):
    """Asocia cada cara detectada con el seguimiento más cercano; crea o descarta seguimientos."""
    used = set()
    for kp in faces:
        mid = (kp["left_eye"] + kp["right_eye"]) / 2
        best, bd = None, 1e9
        for i, tr in enumerate(tracks):
            d = np.linalg.norm(tr.center() - mid)
            if i not in used and d < bd and d < 2 * tr.iod():
                best, bd = i, d
        if best is None:
            tr = Track()
            in_use = [t_.mask for t_ in tracks]
            tr.mask = min(range(len(MASKS)), key=lambda m: (in_use.count(m), m))
            tracks.append(tr)
            best = len(tracks) - 1
        tracks[best].update(kp, t)
        used.add(best)
    for i, tr in enumerate(tracks):
        if i not in used:
            tr.miss += 1
    tracks[:] = [tr for tr in tracks if tr.miss <= HOLD]

Bucle de ejecución principal

In [4]:
face = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=FACE_MODEL),
    running_mode=vision.RunningMode.VIDEO, num_faces=MAX_FACES))

# Índices de la malla facial de Mediapipe
EYE_A, EYE_B = (33, 133), (362, 263)
NOSE_I, MOUTH_A, MOUTH_B = 1, 61, 291


def detect(frame, t):
    """Lista de dicts con los 5 landmarks de cada cara, sacados de la malla facial de Mediapipe."""
    res = face.detect_for_video(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), int(t * 1000))
    H, W = frame.shape[:2]
    out = []
    for lm in res.face_landmarks:
        p = lambda i: np.array([lm[i].x * W, lm[i].y * H])
        out.append({"left_eye": (p(EYE_A[0]) + p(EYE_A[1])) / 2,
                    "right_eye": (p(EYE_B[0]) + p(EYE_B[1])) / 2,
                    "nose": p(NOSE_I), "mouth_left": p(MOUTH_A), "mouth_right": p(MOUTH_B)})
    return out


cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

tracks = []
writer = None
frame_i = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    t = time.perf_counter()

    if frame_i % DETECT_EVERY == 0:
        faces = sorted(detect(frame, t), key=lambda kp: -np.linalg.norm(kp["left_eye"] - kp["right_eye"]))[:MAX_FACES]
        update_tracks(tracks, faces, t)                                    # las MAX_FACES caras más grandes
    frame_i += 1

    for tr in tracks:
        kp = tr.kp(t)
        tr.gain = 0.9 * tr.gain + 0.1 * brightness_gain(frame, kp)
        apply_filter(frame, kp, tr.mask, tr.gain)

    if writer is not None:
        writer.write(frame)
        cv2.circle(frame, (20, 20), 8, (0, 0, 255), -1)
    cv2.imshow("Filtro", frame)

    tecla = cv2.waitKey(20) & 0xFF
    if tecla in (ord("q"), 27):   # q o Esc
        break
    elif tecla == ord("m"): # Con la tecla m se puede cambiar la máscara
        for tr in tracks:
            tr.mask = (tr.mask + 1) % len(MASKS)

if writer is not None:
    writer.release()
face.close()
cap.release()
cv2.destroyAllWindows()